# 2. Understand the model and verify the data construction

This notebook explains the common response model, verifies the bundled source files, and independently reconstructs the frozen training/test split. It also displays the exact three-by-three construction used in the letter.

No model fitting or data download takes place. Run the cells in order after installing the dependencies listed in the repository README. The analysis reads only the bundled snapshot and its recorded design.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works when Jupyter starts in the repository root or a directory below it.
working_directory = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / "idealpoint").is_dir()
     and (candidate / "reference").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the replication repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
REFERENCE = ROOT / "reference"
os.environ.setdefault("MPLBACKEND", "Agg")
cache_directory = ROOT / "outputs" / "_matplotlib"
cache_directory.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_directory))

import numpy as np
import pandas as pd
from IPython.display import Image, JSON, Markdown, display

print("Repository:", ROOT.name)


## 1. Separate distance from the response function

The single-location models compared here use

$$\Pr(Y_{ij}=1\mid\mathcal P)=\operatorname{logit}^{-1}(\eta_{ij}),\qquad
\eta_{ij}=a_i+b_j-\gamma\lVert z_i-w_j\rVert_2^q.$$

The actor and item effects are $a_i$ and $b_j$. Their positions are $z_i$ and $w_j$. The exponent is $q=1$ for unsquared distance and $q=2$ for squared distance. In both specifications, the coordinate distance is Euclidean.

At fixed intercepts and scale, unsquared distance gives a constant change in log odds per unit of separation. Squared distance gives a larger change when the item is farther away. This change in the response function does not remove the Euclidean metric on the coordinates.

The empirical illustration uses two dimensions, $\gamma=1$, independent $N(0,3^2)$ additive effects, and independent $N(0,s^2)$ coordinate components. The values of $s$ are 0.5, 1, and 2. These are approximate joint MAP fits. Matching parameter priors across exponents does not match their induced priors on responses.

This comparison concerns one location per item. Conventional roll-call utility models instead compare Yea and Nay alternatives. The letter discusses that distinction separately.


## 2. Verify the source snapshot

The source is the public Voteview snapshot bundled with the repository. The stored SHA-256 hashes identify the exact files used. A mismatch indicates a changed input and should be resolved before interpreting later checks.

NOMINATE scores and party labels are not predictors in the fitted models. Party labels are used after estimation for display orientation and descriptive comparisons.


In [ ]:
import hashlib

design = json.loads((DATA / "fit_comparison_design.json").read_text())
source_checks = []
for filename, source in design["sources"].items():
    digest = hashlib.sha256((DATA / filename).read_bytes()).hexdigest()
    source_checks.append({
        "File": filename,
        "SHA-256 matches": digest == source["sha256"],
        "Recorded source": source["url"],
    })
source_table = pd.DataFrame(source_checks)
display(source_table)
assert source_table["SHA-256 matches"].all(), "A source file differs from the frozen snapshot."


## 3. Build the member-by-roll-call matrix

Keep Senate members and Senate roll calls. A vote is observed only when its cast code is 1 (Yea) or 6 (Nay). Other codes remain missing. Membership in the Senate member file is also enforced, excluding presidential records that can appear in the vote file.

Members are retained if they have observed Yea/Nay votes on at least 80% of the original roll calls. Sorting the identifiers before drawing the split is part of the frozen design.


In [ ]:
members_raw = pd.read_csv(DATA / "S114_members.csv")
rollcalls_raw = pd.read_csv(DATA / "S114_rollcalls.csv")
votes_raw = pd.read_csv(DATA / "S114_votes.csv")

senate_members = members_raw.loc[members_raw["chamber"].eq("Senate")].drop_duplicates("icpsr")
member_ids = np.sort(senate_members["icpsr"].to_numpy(dtype=int))
roll_ids = np.sort(rollcalls_raw.loc[
    rollcalls_raw["chamber"].eq("Senate"), "rollnumber"
].unique().astype(int))
initial_members, initial_rollcalls = len(member_ids), len(roll_ids)

eligible = votes_raw.loc[
    votes_raw["chamber"].eq("Senate")
    & votes_raw["cast_code"].isin([1, 6])
    & votes_raw["icpsr"].isin(member_ids)
].copy()
assert not eligible.duplicated(["icpsr", "rollnumber"]).any(), "Duplicate observed votes."
assert eligible["rollnumber"].isin(roll_ids).all(), "A vote has no matching Senate roll call."

member_index = {identifier: i for i, identifier in enumerate(member_ids)}
roll_index = {identifier: j for j, identifier in enumerate(roll_ids)}
row_index = eligible["icpsr"].map(member_index).to_numpy(dtype=int)
column_index = eligible["rollnumber"].map(roll_index).to_numpy(dtype=int)
outcomes = np.full((len(member_ids), len(roll_ids)), np.nan)
outcomes[row_index, column_index] = eligible["cast_code"].eq(1).to_numpy(dtype=float)

retain_member = np.isfinite(outcomes).sum(axis=1) >= np.ceil(0.8 * len(roll_ids))
outcomes = outcomes[retain_member]
member_ids = member_ids[retain_member]
print(f"Original grid: {initial_members} members by {initial_rollcalls} roll calls")
print(f"Members retained: {len(member_ids)}")


## 4. Recreate the split and apply the training-outcome filter

Independent uniform draws assign approximately 20% of the observed entries to the held-out set. The seed is fixed before the roll-call filter.

A retained roll call must have observed Yea/Nay votes from at least 80% of retained members and at least 50 training votes. Its training Yea share must lie between 0.1 and 0.9. The outcome-balance filter uses only training outcomes. Participation filters concern which entries are observed.


In [ ]:
observed = np.isfinite(outcomes)
rng = np.random.default_rng(design["split"]["seed"])
draws = rng.random(outcomes.shape)
test_fraction = design["split"]["test_probability"]
train = observed & (draws >= test_fraction)
test = observed & (draws < test_fraction)

training_count = train.sum(axis=0)
training_yeas = np.where(train, outcomes, 0).sum(axis=0)
training_share = np.divide(
    training_yeas, training_count,
    out=np.zeros_like(training_yeas), where=training_count > 0,
)
retain_roll = (
    (observed.sum(axis=0) >= np.ceil(0.8 * len(member_ids)))
    & (training_count >= 50)
    & (training_share >= 0.1)
    & (training_share <= 0.9)
)
outcomes = outcomes[:, retain_roll]
train = train[:, retain_roll]
test = test[:, retain_roll]
roll_ids = roll_ids[retain_roll]

# Missing entries are stored as zero only for array storage. The masks exclude
# them from both fitting and scoring, so they are never treated as Nay votes.
rebuilt = {
    "y": np.nan_to_num(outcomes),
    "train": train,
    "test": test,
    "member_ids": member_ids,
    "roll_ids": roll_ids,
}
assert not np.any(train & test)
assert np.array_equal(train | test, np.isfinite(outcomes))


## 5. Compare every frozen array

This comparison checks outcomes, both masks, and identifier ordering. The summary also verifies the reported sample sizes. No fitted coordinates or held-out outcomes are used to select a model here.


In [ ]:
array_checks = []
with np.load(DATA / "fit_comparison_split.npz", allow_pickle=False) as saved:
    for name, rebuilt_array in rebuilt.items():
        array_checks.append({
            "Array": name,
            "Shape": str(rebuilt_array.shape),
            "Exact match": np.array_equal(rebuilt_array, saved[name]),
        })
array_table = pd.DataFrame(array_checks)
display(array_table)
assert array_table["Exact match"].all(), "The rebuilt split differs from the stored split."

rebuilt_counts = {
    "n_members": len(member_ids),
    "n_rollcalls": len(roll_ids),
    "n_observed": int((train | test).sum()),
    "n_train": int(train.sum()),
    "n_test": int(test.sum()),
    "n_missing": int((~(train | test)).sum()),
}
count_table = pd.DataFrame({
    "Rebuilt": rebuilt_counts,
    "Recorded": {key: design[key] for key in rebuilt_counts},
})
display(count_table)
assert count_table["Rebuilt"].equals(count_table["Recorded"])


## 6. Construct the exact example

Put three actors and three items at $(-1,0,1)$, set all additive effects to zero, and use $\gamma=1$. For each exponent, center the log-odds matrix by subtracting its row and column means and adding its grand mean.

Under squared distance, expanding the square absorbs the coordinate norms into unrestricted additive effects. The centered matrix is a product of the centered coordinate matrices and has rank at most the dimension. In this one-dimensional example the ranks are two for unsquared distance and one for squared distance.

Rank establishes that this unsquared-distance matrix cannot be represented by any one-dimensional squared-distance model. The converse requires the interval-overlap argument in the supplementary material. Rank alone does not prove both directions.


In [ ]:
from fractions import Fraction
from idealpoint.checks import exact_checks

positions = [-1, 0, 1]
for q in (1, 2):
    eta = [[Fraction(-abs(x - w) ** q) for w in positions] for x in positions]
    row_means = [sum(row) / 3 for row in eta]
    column_means = [sum(eta[i][j] for i in range(3)) / 3 for j in range(3)]
    grand_mean = sum(row_means) / 3
    centered = [[
        str(eta[i][j] - row_means[i] - column_means[j] + grand_mean)
        for j in range(3)
    ] for i in range(3)]
    display(Markdown(f"**Centered log odds, q = {q}**"))
    display(pd.DataFrame(centered, index=positions, columns=positions))

exact_report = exact_checks()
display(JSON(exact_report, expanded=False))
assert exact_report["passed"], "An exact arithmetic check failed."


## Scope of the construction

Each one-dimensional family permits probability patterns that the other cannot represent, even after re-estimating positions and unrestricted intercepts. This exact result does not establish the same noncontainment in every dimension, nor does it rank approximate fits or predictive performance.

The Senate illustration answers a separate empirical question about the fitted procedures on a fixed dataset and split. Return to `1_reproduce_manuscript.ipynb` for the saved results or continue to `3_refit_senate.ipynb` for new estimates.
